<a href="https://colab.research.google.com/github/Leonanda1013/DataLoverz/blob/main/Competition/GammaFest_IPB/2_Model_PoissonRegression_gamma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
import numpy as np

In [ ]:
df_train = pd.read_csv('train_clean.csv')
df_test = pd.read_csv('test_clean.csv')

In [ ]:
train = df_train.copy()
test = df_test.copy()

test_id = test['Id']

train = train.drop(columns=['id'], errors='ignore')
test = test.drop(columns=['id'], errors='ignore')

In [ ]:
target_cols = ['team_goals', 'opp_goals']

X = train.drop(columns=target_cols)
y = train[target_cols]

In [ ]:
cat_cols = X.select_dtypes(include=['object', 'category']).columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

In [ ]:
base_model = XGBRegressor(
    objective='count:poisson',   # 🔥 penting!
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42
)

model = MultiOutputRegressor(base_model)

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', model)
])

In [ ]:
pipeline.fit(X, y)

In [ ]:
preds = pipeline.predict(test)

In [ ]:
submission = pd.DataFrame({
    'Id': test_id,
    'team_goals': np.round(preds[:, 0]).astype(int),
    'opp_goals': np.round(preds[:, 1]).astype(int)
})

In [ ]:
submission.to_csv('submission_gmc_poissonRegression.csv',index = False)

In [ ]:
submission.head()